# EDA météo — Fourcasters

Ce notebook explore les données météo utilisées dans le projet Fourcasters.

Les données viennent d'Open-Meteo et sont stockées dans BigQuery après les transformations dbt.
L'objectif est de vérifier la qualité des données et de comprendre les grandes tendances météo avant de les utiliser dans Power BI et dans le modèle.


## Préparation


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.cloud import bigquery

from fourcasters_dbt.configuration import (
    PROJET_GCP,
    DATASET_ANALYSE,
    configurer_google_cloud,
)

configurer_google_cloud()

DATASET = DATASET_ANALYSE
client = bigquery.Client(project=PROJET_GCP)

def lire_requete(sql):
    return client.query(sql).to_dataframe(create_bqstorage_client=False)


ModuleNotFoundError: No module named 'matplotlib'

La connexion utilise directement la configuration du projet. Les requêtes sont faites dans BigQuery pour éviter de charger plusieurs millions de lignes en mémoire.


## 1. Taille du jeu de données et période couverte


In [3]:
resume = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  COUNT(DISTINCT date) AS jours,
  COUNT(DISTINCT code_insee) AS points_meteo,
  COUNT(DISTINCT numero_departement) AS departements,
  MIN(date) AS date_debut,
  MAX(date) AS date_fin
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
""")

resume


NameError: name 'lire_requete' is not defined

Cette première vérification donne la taille réelle du jeu de données. On s'attend à retrouver les 360 points météo du référentiel, sur une période qui commence en 2000.


## 2. Valeurs manquantes sur les variables principales


In [ ]:
qualite = lire_requete(f"""
SELECT
  COUNT(*) AS lignes,
  ROUND(100 * COUNTIF(temperature_moyenne IS NULL) / COUNT(*), 3) AS pct_temperature_manquante,
  ROUND(100 * COUNTIF(humidite_moyenne IS NULL) / COUNT(*), 3) AS pct_humidite_manquante,
  ROUND(100 * COUNTIF(precipitations_totales IS NULL) / COUNT(*), 3) AS pct_precipitations_manquantes,
  ROUND(100 * COUNTIF(rafale_vent_maximale IS NULL) / COUNT(*), 3) AS pct_rafales_manquantes,
  ROUND(100 * COUNTIF(deficit_pression_vapeur_maximal IS NULL) / COUNT(*), 3) AS pct_vpd_manquant
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
""")

qualite


Ces cinq variables sont importantes pour l'analyse du danger incendie. Si les pourcentages sont très faibles ou nuls, le jeu de données est suffisamment complet pour les analyses suivantes.


## 3. Couverture quotidienne des 360 points


In [ ]:
couverture = lire_requete(f"""
SELECT
  date,
  COUNT(DISTINCT code_insee) AS nombre_points
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
GROUP BY date
ORDER BY date
""")

display(couverture["nombre_points"].describe())

jours_incomplets = couverture[couverture["nombre_points"] != 360]
print("Nombre de jours incomplets :", len(jours_incomplets))
display(jours_incomplets.tail(15))

couverture.plot(x="date", y="nombre_points", figsize=(10, 4), legend=False)
plt.axhline(360, linestyle="--")
plt.title("Nombre de points météo disponibles par jour")
plt.ylabel("Nombre de points")
plt.xlabel("")
plt.show()


Une couverture stable à 360 signifie que tous les points sont présents chaque jour. Les éventuels jours incomplets sont à repérer avant de comparer des périodes ou des territoires.


## 4. Statistiques générales des principales variables météo


In [ ]:
statistiques = lire_requete(f"""
WITH valeurs AS (
  SELECT variable, valeur
  FROM `{PROJET_GCP}.{DATASET}.fact_meteo`,
  UNNEST([
    STRUCT('Température moyenne (°C)' AS variable, CAST(temperature_moyenne AS FLOAT64) AS valeur),
    STRUCT('Température maximale (°C)', CAST(temperature_maximale AS FLOAT64)),
    STRUCT('Humidité moyenne (%)', CAST(humidite_moyenne AS FLOAT64)),
    STRUCT('Précipitations (mm)', CAST(precipitations_totales AS FLOAT64)),
    STRUCT('Rafale maximale (km/h)', CAST(rafale_vent_maximale AS FLOAT64)),
    STRUCT('VPD maximal (kPa)', CAST(deficit_pression_vapeur_maximal AS FLOAT64))
  ])
)
SELECT
  variable,
  COUNT(valeur) AS valeurs,
  ROUND(AVG(valeur), 2) AS moyenne,
  ROUND(STDDEV(valeur), 2) AS ecart_type,
  ROUND(APPROX_QUANTILES(valeur, 4)[OFFSET(1)], 2) AS q1,
  ROUND(APPROX_QUANTILES(valeur, 4)[OFFSET(2)], 2) AS mediane,
  ROUND(APPROX_QUANTILES(valeur, 4)[OFFSET(3)], 2) AS q3,
  ROUND(MIN(valeur), 2) AS minimum,
  ROUND(MAX(valeur), 2) AS maximum
FROM valeurs
GROUP BY variable
ORDER BY variable
""")

statistiques


Ce tableau donne une vue simple de la distribution des variables : moyenne, dispersion, quartiles et valeurs extrêmes. Les minimums et maximums permettent aussi de repérer rapidement une valeur qui semblerait incohérente.


## 5. Évolution annuelle de la température et des précipitations


In [ ]:
par_annee = lire_requete(f"""
SELECT
  EXTRACT(YEAR FROM date) AS annee,
  ROUND(AVG(temperature_moyenne), 2) AS temperature_moyenne,
  ROUND(AVG(precipitations_totales), 2) AS precipitation_moyenne_journaliere
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
GROUP BY annee
ORDER BY annee
""")

display(par_annee)

par_annee.plot(x="annee", y="temperature_moyenne", marker="o", figsize=(10, 4), legend=False)
plt.title("Température moyenne par année")
plt.ylabel("Température moyenne (°C)")
plt.xlabel("Année")
plt.show()

par_annee.plot(x="annee", y="precipitation_moyenne_journaliere", marker="o", figsize=(10, 4), legend=False)
plt.title("Précipitations moyennes par jour selon l'année")
plt.ylabel("Précipitations (mm)")
plt.xlabel("Année")
plt.show()


Les courbes permettent de voir si certaines années ressortent comme plus chaudes ou plus humides. Il faut surtout regarder les tendances générales et éviter de tirer une conclusion à partir d'une seule année.


## 6. Saisonnalité de la météo


In [ ]:
par_mois = lire_requete(f"""
SELECT
  EXTRACT(MONTH FROM date) AS mois,
  ROUND(AVG(temperature_moyenne), 2) AS temperature_moyenne,
  ROUND(AVG(humidite_moyenne), 2) AS humidite_moyenne,
  ROUND(AVG(precipitations_totales), 2) AS precipitations_moyennes,
  ROUND(AVG(rafale_vent_maximale), 2) AS rafale_moyenne
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
GROUP BY mois
ORDER BY mois
""")

display(par_mois)

par_mois.plot(x="mois", y="temperature_moyenne", marker="o", figsize=(9, 4), legend=False)
plt.title("Température moyenne selon le mois")
plt.ylabel("Température (°C)")
plt.xticks(range(1, 13))
plt.show()

par_mois.plot(x="mois", y="humidite_moyenne", marker="o", figsize=(9, 4), legend=False)
plt.title("Humidité moyenne selon le mois")
plt.ylabel("Humidité (%)")
plt.xticks(range(1, 13))
plt.show()

par_mois.plot(x="mois", y="precipitations_moyennes", marker="o", figsize=(9, 4), legend=False)
plt.title("Précipitations moyennes selon le mois")
plt.ylabel("Précipitations (mm)")
plt.xticks(range(1, 13))
plt.show()


La saisonnalité doit être nette : températures plus élevées en été, avec des variations d'humidité et de pluie selon les mois. C'est important car le danger incendie est lui aussi très saisonnier.


## 7. Comparaison entre les départements


In [ ]:
departements = lire_requete(f"""
SELECT
  numero_departement,
  ANY_VALUE(departement) AS departement,
  ROUND(AVG(temperature_moyenne), 2) AS temperature_moyenne,
  ROUND(AVG(humidite_moyenne), 2) AS humidite_moyenne,
  ROUND(AVG(precipitations_moyennes), 2) AS precipitations_moyennes,
  ROUND(AVG(rafale_vent_maximale), 2) AS rafale_moyenne,
  COUNT(DISTINCT date) AS jours
FROM `{PROJET_GCP}.{DATASET}.int_meteo_departement_jour`
GROUP BY numero_departement
""")

print("Départements les plus chauds")
display(departements.sort_values("temperature_moyenne", ascending=False).head(10))

print("Départements les plus secs en moyenne")
display(departements.sort_values("precipitations_moyennes").head(10))

print("Départements les plus venteux")
display(departements.sort_values("rafale_moyenne", ascending=False).head(10))


Les écarts entre départements montrent qu'on ne peut pas résumer la météo française avec une seule moyenne nationale. La chaleur, la pluie et le vent n'ont pas la même importance selon les territoires.


## 8. Relations entre les variables météo


In [ ]:
echantillon = lire_requete(f"""
SELECT
  temperature_moyenne,
  temperature_maximale,
  humidite_moyenne,
  precipitations_moyennes,
  rafale_vent_maximale,
  deficit_pression_vapeur_maximal
FROM `{PROJET_GCP}.{DATASET}.int_meteo_departement_jour`
WHERE MOD(
  ABS(FARM_FINGERPRINT(CONCAT(CAST(date AS STRING), '|', numero_departement))),
  20
) = 0
""")

correlations = echantillon.corr(numeric_only=True)
display(correlations.round(2))

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlations, vmin=-1, vmax=1)
ax.set_xticks(range(len(correlations.columns)))
ax.set_xticklabels(correlations.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlations.index)))
ax.set_yticklabels(correlations.index)
plt.colorbar(image, ax=ax)
plt.title("Corrélations entre les variables météo")
plt.tight_layout()
plt.show()


Les corrélations permettent de repérer les variables qui évoluent souvent ensemble. Par exemple, la température et le VPD peuvent être liés, alors que l'humidité évolue souvent dans le sens opposé. Une corrélation ne prouve pas une causalité.


## 9. Journées météo les plus extrêmes


In [ ]:
extremes = lire_requete(f"""
SELECT
  m.date,
  m.numero_departement,
  m.departement,
  ROUND(m.temperature_maximale, 1) AS temperature_maximale,
  ROUND(m.humidite_moyenne, 1) AS humidite_moyenne,
  ROUND(m.rafale_vent_maximale, 1) AS rafale_vent_maximale,
  ROUND(m.precipitations_moyennes, 1) AS precipitations_moyennes
FROM `{PROJET_GCP}.{DATASET}.int_meteo_departement_jour` AS m
""")

print("Journées les plus chaudes")
display(extremes.sort_values("temperature_maximale", ascending=False).head(10))

print("Journées les plus sèches en humidité")
display(extremes.sort_values("humidite_moyenne").head(10))

print("Journées avec les plus fortes rafales")
display(extremes.sort_values("rafale_vent_maximale", ascending=False).head(10))


Ces valeurs extrêmes sont utiles car le danger incendie dépend souvent de combinaisons de chaleur, d'air sec et de vent. Elles doivent cependant être replacées dans leur date et leur département.


## 10. Codes météo les plus fréquents


In [ ]:
codes_meteo = lire_requete(f"""
SELECT
  code_meteo,
  COUNT(*) AS observations,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM `{PROJET_GCP}.{DATASET}.fact_meteo`
GROUP BY code_meteo
ORDER BY observations DESC
""")

display(codes_meteo.head(15))

codes_meteo.head(15).plot(
    x="code_meteo",
    y="pct",
    kind="bar",
    figsize=(9, 4),
    legend=False,
)
plt.title("Codes météo les plus fréquents")
plt.ylabel("Part des observations (%)")
plt.xlabel("Code météo WMO")
plt.show()


Les codes météo donnent une vision descriptive des conditions rencontrées. Ils sont surtout utiles pour l'EDA et Power BI ; pour le modèle, les variables numériques détaillées sont plus informatives.


## Bilan

À la fin de cet EDA, je vérifie surtout trois choses :

- la couverture des 360 points dans le temps ;
- la cohérence des principales variables météo ;
- les différences de saison et de territoire.

Ces étapes permettent de mieux comprendre les données avant de les croiser avec le niveau de danger incendie.
